In [1]:
# =========================
# STEP 1: IMPORTS & OVERVIEW
# =========================
# This script:
# 1. Loads your EPC dataset
# 2. Iterates through each BUILDING_REFERENCE_NUMBER
# 3. Finds the corresponding energy_results.csv file
# 4. Extracts the maximum value from the "electricity_heat_hp" column
# 5. Writes this value into a new (or replaced) column called "PEAK_HP_LOAD"
# 6. Saves the updated dataset to a new CSV file
#
# Designed for robustness:
# - Handles missing folders/files
# - Handles missing columns
# - Logs issues for troubleshooting

import os
import pandas as pd

In [2]:
# =========================
# STEP 2: DEFINE FILE PATHS
# =========================

epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_cap_cost_updated.csv"
baseline_root = "/Users/jakegehrung/Desktop/data_raw/baseline_models"
output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_hp_peak_load.csv"

In [3]:
# =========================
# STEP 3: LOAD EPC DATASET
# =========================

epc_df = pd.read_csv(epc_path)

# Ensure BUILDING_REFERENCE_NUMBER is treated as string for path consistency
epc_df["BUILDING_REFERENCE_NUMBER"] = epc_df["BUILDING_REFERENCE_NUMBER"].astype(str)

# Initialise or replace PEAK_HP_LOAD column
epc_df["PEAK_HP_LOAD"] = None

print(f"EPC dataset loaded: {len(epc_df)} rows")

EPC dataset loaded: 117 rows


In [4]:
# =========================
# STEP 4: DEFINE FUNCTION TO EXTRACT PEAK LOAD
# =========================

def get_peak_hp_load(building_id):
    """
    Returns the max electricity_heat_hp value for a given building ID.
    Returns None if file/column is missing or unreadable.
    """
    energy_file = os.path.join(
        baseline_root,
        building_id,
        "TOTAL",
        "energy_results.csv"
    )
    
    if not os.path.exists(energy_file):
        print(f"[WARNING] Missing file: {energy_file}")
        return None
    
    try:
        df = pd.read_csv(energy_file)
        
        if "electricity_heat_hp" not in df.columns:
            print(f"[WARNING] Column missing in: {energy_file}")
            return None
        
        return df["electricity_heat_hp"].max()
    
    except Exception as e:
        print(f"[ERROR] Failed to process {energy_file}: {e}")
        return None

In [5]:
# =========================
# STEP 5: PROCESS ALL BUILDINGS
# =========================

for idx, row in epc_df.iterrows():
    building_id = row["BUILDING_REFERENCE_NUMBER"]
    
    peak_value = get_peak_hp_load(building_id)
    
    epc_df.at[idx, "PEAK_HP_LOAD"] = peak_value

print("Processing complete.")

Processing complete.


In [6]:
# =========================
# STEP 6: SAVE OUTPUT FILE
# =========================

epc_df.to_csv(output_path, index=False)

print(f"Output saved to: {output_path}")

Output saved to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_hp_peak_load.csv
